In [2]:
import agentscope

print(agentscope.__version__)

1.0.7


### <center>2. AgentScope模型接入流程

#### 2.1 阿里云模型接入

In [3]:
from agentscope.model import DashScopeChatModel

In [7]:
model = DashScopeChatModel(
    model_name="qwen-max",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    stream=False,
)

In [8]:
messages=[{"role": "user", "content": "你好，好久不见！"},]

In [14]:
res = await model(messages = messages)

In [15]:
res

ChatResponse(content=[{'type': 'text', 'text': '你好！很高兴再次与您交谈。有什么可以帮助您的吗？'}], id='2025-11-10 17:34:56.097_472b35', created_at='2025-11-10 17:34:56.097', type='chat', usage=ChatUsage(input_tokens=13, output_tokens=13, time=1.234083, type='chat'), metadata=None)

In [16]:
res.content

[{'type': 'text', 'text': '你好！很高兴再次与您交谈。有什么可以帮助您的吗？'}]

#### 2.2 不同模型采用不同基础库进行接入

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110173712231.png" alt="image-20251110173712231" style="zoom:50%;" />

In [4]:
from agentscope.model import OpenAIChatModel

In [18]:
OpenAIChatModel?

Init signature:
OpenAIChatModel(
    model_name: str,
    api_key: str | None = None,
    stream: bool = True,
    reasoning_effort: Optional[Literal['low', 'medium', 'high']] = None,
    organization: str = None,
    client_args: dict = None,
    generate_kwargs: dict[str, typing.Union[str, int, float, bool, NoneType, list['JSONSerializableObject'], dict[str, 'JSONSerializableObject']]] | None = None,
) -> None
Docstring:      The OpenAI chat model class.
Init docstring:
Initialize the openai client.

Args:
    model_name (`str`, default `None`):
        The name of the model to use in OpenAI API.
    api_key (`str`, default `None`):
        The API key for OpenAI API. If not specified, it will
        be read from the environment variable `OPENAI_API_KEY`.
    stream (`bool`, default `True`):
        Whether to use streaming output or not.
    reasoning_effort (`Literal["low", "medium", "high"] | None`,             optional):
        Reasoning effort, supported for o3, o4, etc. Pleas

In [22]:
# import os
# os.environ['HTTP_PROXY'] = 'http://127.0.0.1:10080'
# os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:10080'
from dotenv import load_dotenv 
load_dotenv(override=True)

True

In [23]:
model = OpenAIChatModel(
    model_name="gpt-5",
    api_key=os.environ["OPENAI_API_KEY"],
    stream=False,
)

In [24]:
res = await model(messages = messages)

In [25]:
res

ChatResponse(content=[{'type': 'text', 'text': '嗨！确实好久不见了，最近过得怎么样？有什么新鲜事想聊，或者需要我帮你做点什么吗？'}], id='2025-11-10 17:51:11.679_724f66', created_at='2025-11-10 17:51:11.679', type='chat', usage=ChatUsage(input_tokens=12, output_tokens=170, time=6.148201, type='chat'), metadata=None)

### <center>3. AgentScope中Agent创建流程

- 核心API：ReActAgent

In [5]:
from agentscope.agent import ReActAgent

In [65]:
ReActAgent?

Init signature:
ReActAgent(
    name: str,
    sys_prompt: str,
    model: agentscope.model._model_base.ChatModelBase,
    formatter: agentscope.formatter._formatter_base.FormatterBase,
    toolkit: agentscope.tool._toolkit.Toolkit | None = None,
    memory: agentscope.memory._memory_base.MemoryBase | None = None,
    long_term_memory: agentscope.memory._long_term_memory_base.LongTermMemoryBase | None = None,
    long_term_memory_mode: Literal['agent_control', 'static_control', 'both'] = 'both',
    enable_meta_tool: bool = False,
    parallel_tool_calls: bool = False,
    knowledge: agentscope.rag._knowledge_base.KnowledgeBase | list[agentscope.rag._knowledge_base.KnowledgeBase] | None = None,
    enable_rewrite_query: bool = True,
    plan_notebook: agentscope.plan._plan_notebook.PlanNotebook | None = None,
    print_hint_msg: bool = False,
    max_iters: int = 10,
) -> None
Docstring:     
A ReAct agent implementation in AgentScope, which supports

- Realtime steering
- API-based (p

相关参数解释如下：

| 参数 | 描述 |
|------|------|
| **name**（必需） | 智能体的名称 |
| **sys_prompt**（必需） | 智能体的系统提示 |
| **model**（必需） | 智能体用于生成响应的模型 |
| **formatter**（必需） | 提示构建策略，应与使用的模型保持一致 |
| **toolkit** | 用于注册/调用工具函数的工具模块 |
| **memory** | 用于存储对话历史的短期记忆 |
| **long_term_memory** | 长期记忆 |
| **long_term_memory_mode** | 长期记忆的管理模式：<br>• **agent_control**：允许智能体通过工具函数自己控制长期记忆<br>• **static_control**：在每次回复（reply）的开始/结束时，会自动从长期记忆中检索/记录信息<br>• **both**：同时激活上述两种模式 |
| **enable_meta_tool** | 是否启用元工具（Meta tool），即允许智能体自主管理工具函数 |
| **parallel_tool_calls** | 是否允许并行进行工具调用 |
| **max_iters** | 智能体生成响应的最大迭代次数 |
| **plan_notebook** | 计划模块，允许智能体制定和管理计划与子任务 |
| **print_hint_msg** | 是否在终端打印由 `plan_notebook` 生成的提示消息 |


- 创建Agent极简流程

In [6]:
from agentscope.agent import ReActAgent
from agentscope.message import Msg
from agentscope.formatter import DashScopeChatFormatter
from agentscope.memory import InMemoryMemory

In [60]:
xiaozhi = ReActAgent(
    name="小智",
    sys_prompt="你是一个名为小智的智能助手",
    model=DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=False,
        enable_thinking=False,
    ),
    formatter=DashScopeChatFormatter(),
    memory=InMemoryMemory(),
)

In [61]:
msg = Msg(
    name="user1",
    content="你好，好久不见，请介绍下你自己。",
    role="user",
)

实际运行效果如下：

In [62]:
res = await xiaozhi(msg)

小智: 你好！我是小智，一个能够帮助你处理多种任务的智能助手。无论你需要信息查询、日程管理还是其他帮助，我都会尽全力提供支持。有什么我可以帮到你的吗？


In [63]:
res

Msg(id='8cMwUoEaR7xKBs8Q87ZUre', name='小智', content='你好！我是小智，一个能够帮助你处理多种任务的智能助手。无论你需要信息查询、日程管理还是其他帮助，我都会尽全力提供支持。有什么我可以帮到你的吗？', role='assistant', metadata=None, timestamp='2025-11-10 18:16:12.825', invocation_id='None')

- Message对象说明

&emsp;&emsp;其中需要注意的是，Agent输入和输出都是Message格式，二AgentScope 中的 **消息（Message）** 是框架的核心概念之一，用于支持多模态数据、工具 API、信息存储/交换和提示构建。  一个完整的消息对象由以下四个字段组成：

| 字段 | 类型 | 描述 |
|------|------|------|
| **name** | `str` | 消息发送者的名称或身份标识。 |
| **role** | `Literal["system", "assistant", "user"]` | 消息发送者的角色，必须是 `"system"`、`"assistant"` 或 `"user"` 之一。 |
| **content** | `str` \| `list[ContentBlock]` | 消息的主要内容，可以是文本字符串，也可以是多模态内容块（如图片、音频、结构化数据等）的列表。 |
| **metadata** | `dict[str, JSONSerializableObject]` \| `None` | 包含额外元数据的字典，常用于结构化输出（如模型生成的标签、预测结果或自定义字段）。 |

In [66]:
(msg, res)

(Msg(id='5TerRaU5o6zrdwGgjarrJQ', name='user1', content='你好，好久不见，请介绍下你自己。', role='user', metadata=None, timestamp='2025-11-10 18:16:09.222', invocation_id='None'),
 Msg(id='8cMwUoEaR7xKBs8Q87ZUre', name='小智', content='你好！我是小智，一个能够帮助你处理多种任务的智能助手。无论你需要信息查询、日程管理还是其他帮助，我都会尽全力提供支持。有什么我可以帮到你的吗？', role='assistant', metadata=None, timestamp='2025-11-10 18:16:12.825', invocation_id='None'))

- DashScopeChatFormatter提示词格式化库说明

&emsp;&emsp;AgentScope 中的格式化器（formatter）模块主要负责一下三个事项：

- 将 Msg 对象转换为不同 LLM API 要求的格式，

- （可选）截断消息以适应 max_token 的限制，

- （可选）执行提示工程，例如对长对话进行总结。

&emsp;&emsp;后两个功能是可选的，开发者也可以选择在记忆（memory）或智能体（agent）层面进行处理和实现。在 AgentScope 中，有两种类型的格式化器："ChatFormatter" 和 "MultiAgentFormatter"，它们根据输入消息中的“身份实体”进行区分。

- ChatFormatter：专为标准的用户-助手场景（聊天机器人）设计，使用 role 字段来识别用户和助手。

- MultiAgentFormatter：专为多智能体场景设计，使用 name 字段来识别不同的实体，在格式化的过程中会将多智能体的对话历史合并为单个消息。

| API 提供商 | 用户-助手场景 | 多智能体场景 |
|-------------|----------------|----------------|
| **OpenAI** | `OpenAIChatFormatter` | `OpenAIMultiAgentFormatter` |
| **Anthropic** | `AnthropicChatFormatter` | `AnthropicMultiAgentFormatter` |
| **DashScope** | `DashScopeChatFormatter` | `DashScopeMultiAgentFormatter` |
| **Gemini** | `GeminiChatFormatter` | `GeminiChatFormatter` |
| **Ollama** | `OllamaChatFormatter` | `OllamaMultiAgentFormatter` |
| **DeepSeek** | `DeepSeekChatFormatter` | `DeepSeekMultiAgentFormatter` |
| **vLLM** | `OpenAIFormatter` | `OpenAIFormatter` |

- Agent记忆管理

In [67]:
msg = Msg(
    name="user1",
    content="非常棒，你还记得我上一个问你的问题是什么吗？",
    role="user",
)

In [68]:
res2 = await xiaozhi(msg)

小智: 我们之前的对话中，你问候了我并且请求我自我介绍。这是我们的第一次交流，因此在那之前并没有其他问题。如果有什么特定的问题或帮助是你现在需要的，请告诉我！


In [69]:
res2

Msg(id='MvxVqXgwqjSyWqiyJMdedm', name='小智', content='我们之前的对话中，你问候了我并且请求我自我介绍。这是我们的第一次交流，因此在那之前并没有其他问题。如果有什么特定的问题或帮助是你现在需要的，请告诉我！', role='assistant', metadata=None, timestamp='2025-11-10 18:24:14.004', invocation_id='None')

- 获取历史多轮对话记忆

In [86]:
await xiaozhi.memory.get_memory()

[Msg(id='5TerRaU5o6zrdwGgjarrJQ', name='user1', content='你好，好久不见，请介绍下你自己。', role='user', metadata=None, timestamp='2025-11-10 18:16:09.222', invocation_id='None'),
 Msg(id='avqxMrpG3kFYpJZ5fHQvia', name='小智', content=[{'type': 'tool_use', 'name': 'generate_response', 'input': {'response': '你好！我是小智，一个能够帮助你处理多种任务的智能助手。无论你需要信息查询、日程管理还是其他帮助，我都会尽全力提供支持。有什么我可以帮到你的吗？'}, 'id': 'call_45ad279c2d42451d8bbf0e'}], role='assistant', metadata=None, timestamp='2025-11-10 18:16:12.824', invocation_id='None'),
 Msg(id='Hgm6QhzaiDZGRYjdFkXqrR', name='system', content=[{'type': 'tool_result', 'id': 'call_45ad279c2d42451d8bbf0e', 'name': 'generate_response', 'output': [{'type': 'text', 'text': 'Successfully generated response.'}]}], role='system', metadata=None, timestamp='2025-11-10 18:16:12.824', invocation_id='None'),
 Msg(id='8cMwUoEaR7xKBs8Q87ZUre', name='小智', content='你好！我是小智，一个能够帮助你处理多种任务的智能助手。无论你需要信息查询、日程管理还是其他帮助，我都会尽全力提供支持。有什么我可以帮到你的吗？', role='assistant', metadata=None, timestamp='2025-11-10 18:16

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110183150745.png" alt="image-20251110183150745" style="zoom:50%;" />

- 借助AgentScope进行前端对话与监控

In [9]:
import os

In [10]:
agentscope.init(studio_url="http://localhost:3000")

xiaozhi = ReActAgent(
    name="小智",
    sys_prompt="你是一个名为小智的智能助手",
    model=DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=False,
        enable_thinking=False,
    ),
    formatter=DashScopeChatFormatter(),
    memory=InMemoryMemory(),
)

2025-11-10 18:37:31,366 | INFO    | _user_input:on_connect:194 - Connected to AgentScope Studio at "http://localhost:3000" with run name "4WmqRcuhNCk8vzTw9VqH5x".
2025-11-10 18:37:31,369 | INFO    | _user_input:on_connect:200 - View the run at: http://localhost:3000/dashboard/projects/UnnamedProject_At20251110
Overriding of current TracerProvider is not allowed


> 注，此处可以在init的过程设置project来调整项目的名称，否则会设置为随机字符串。

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110183755134.png" style="zoom:33%;" />

### <center>4. AgentScope消息类型与消息格式化方法

&emsp;&emsp;消息是 AgentScope 中的核心概念，用于支持多模态数据、工具 API、信息存储/交换和提示构建。一般消息格式如下：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110175440967.png" alt="image-20251110175440967" style="zoom:50%;" />

In [12]:
from agentscope.message import (
    Msg,
    Base64Source,
    TextBlock,
    ThinkingBlock,
    ImageBlock,
    AudioBlock,
    VideoBlock,
    ToolUseBlock,
    ToolResultBlock,
    URLSource
)
import json

- 基础文本消息创建方法：通过提供 ``name``、``role`` 和 ``content`` 字段来创建消息对象。

In [13]:
msg = Msg(
    name="Jarvis",
    role="assistant",
    content="你好！我能怎么帮助你？",
)

print(f"发送者的名称: {msg.name}")
print(f"发送者的角色: {msg.role}")
print(f"消息的内容: {msg.content}")

发送者的名称: Jarvis
发送者的角色: assistant
消息的内容: 你好！我能怎么帮助你？


- 多模态消息创建与对话

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110175633205.png" alt="image-20251110175633205" style="zoom:50%;" />

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/ffc998ffffe1af118077b4c51dc75148.png" alt="ffc998ffffe1af118077b4c51dc75148" style="zoom:33%;" />

In [37]:
msg = Msg(
    name="user2",
    role="user",
    content=[
        TextBlock(
            type="text",
            text="请帮我详细介绍下这张图片上的信息",
        ),
        ImageBlock(
           type="image",
           source=URLSource(
               type="url",
               url="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/ffc998ffffe1af118077b4c51dc75148.png"
           )
        )
    ],
)

In [38]:
msg

Msg(id='eKPZJqqoLREurGUJf7ZqfP', name='user2', content=[{'type': 'text', 'text': '请帮我详细介绍下这张图片上的信息'}, {'type': 'image', 'source': {'type': 'url', 'url': 'https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/ffc998ffffe1af118077b4c51dc75148.png'}}], role='user', metadata=None, timestamp='2025-11-10 18:54:54.074', invocation_id='None')

In [31]:
from agentscope.formatter import OpenAIChatFormatter

In [36]:
# import os
# os.environ['HTTP_PROXY'] = 'http://127.0.0.1:10080'
# os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:10080'
from dotenv import load_dotenv 
load_dotenv(override=True)

True

In [33]:
model = OpenAIChatModel(
    model_name="gpt-5",
    api_key=os.environ["OPENAI_API_KEY"],
    stream=False,
)

In [34]:
xiaozhi = ReActAgent(
    name="小智",
    sys_prompt="你是一个名为小智的智能助手",
    model=model,
    formatter=OpenAIChatFormatter(),
    memory=InMemoryMemory(),
)

In [35]:
res = await xiaozhi(msg)

小智: 这是一张“双十一”促销海报，整体红色火热风格，金币、红包、火焰元素，右下有二维码和喇叭。

可识别文字（OCR）如下：
- 爆款课程真底价 · 价保全年
- 双十一火爆开启
- 求职 | 晋升 | 论文 | 项目落地一站式速通
- 抢占优惠&详情咨询，请咨询你的专属助教

需要我：
- 导出为可复制文案/翻译成英文
- 改写更高转化的标题与CTA
- 给出版式与配色优化建议，或生成多版本文案
- 适配不同平台尺寸（如朋友圈、横版、电商主图）吗？

如果你说下目标人群和投放渠道，我可以给出更精准的文案与设计建议。


- 基于Formatter的消息队列裁剪

In [39]:
OpenAIChatFormatter?

Init signature:
OpenAIChatFormatter(
    token_counter: agentscope.token._token_base.TokenCounterBase | None = None,
    max_tokens: int | None = None,
) -> None
Docstring:     
The class used to format message objects into the OpenAI API required
format.
Init docstring:
Initialize the TruncatedFormatterBase.

Args:
    token_counter (`TokenCounterBase | None`, optional):
        A token counter instance used to count tokens in the messages.
        If not provided, the formatter will format the messages
        without considering token limits.
    max_tokens (`int | None`, optional):
        The maximum number of tokens allowed in the formatted
        messages. If not provided, the formatter will not truncate
        the messages.
File:           c:\programdata\anaconda3\envs\agentscope\lib\site-packages\agentscope\formatter\_openai_formatter.py
Type:           ABCMeta
Subclasses:     

In [40]:
from agentscope.token import OpenAITokenCounter

In [62]:
# 创建 token 计数器
token_counter = OpenAITokenCounter(model_name="gpt-4o")

# 创建带截断功能的 Formatter
formatter = OpenAIChatFormatter(
    token_counter=token_counter,
    max_tokens=300
)

In [63]:
messages = await xiaozhi.memory.get_memory()

In [64]:
messages

[Msg(id='SSnWUcPG2wrmTY22dNL2f3', name='user2', content=[{'type': 'image', 'source': {'type': 'url', 'url': 'https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/ffc998ffffe1af118077b4c51dc75148.png'}}], role='user', metadata=None, timestamp='2025-11-10 18:50:48.870', invocation_id='None'),
 Msg(id='nhyDecdFNeKjRBQ4GB49Gs', name='小智', content=[{'id': '6Y7M9rFrR6GwMnkbnH4auR', 'type': 'tool_use', 'name': 'generate_response', 'input': {'response': '这是一张“双十一”促销海报，整体红色火热风格，金币、红包、火焰元素，右下有二维码和喇叭。\n\n可识别文字（OCR）如下：\n- 爆款课程真底价 · 价保全年\n- 双十一火爆开启\n- 求职 | 晋升 | 论文 | 项目落地一站式速通\n- 抢占优惠&详情咨询，请咨询你的专属助教\n\n需要我：\n- 导出为可复制文案/翻译成英文\n- 改写更高转化的标题与CTA\n- 给出版式与配色优化建议，或生成多版本文案\n- 适配不同平台尺寸（如朋友圈、横版、电商主图）吗？\n\n如果你说下目标人群和投放渠道，我可以给出更精准的文案与设计建议。'}}], role='assistant', metadata=None, timestamp='2025-11-10 18:53:15.292', invocation_id='None'),
 Msg(id='PYRjSg7NSNV5BxpwGX6Mvs', name='system', content=[{'type': 'tool_result', 'id': '6Y7M9rFrR6GwMnkbnH4auR', 'name': 'generate_response', 'output': [{'type': 'text', 'text': 'Suc

In [65]:
truncated_msgs = await formatter.format(messages)

In [66]:
truncated_msgs

[{'role': 'assistant',
  'name': '小智',
  'content': [{'type': 'text',
    'text': '这是一张“双十一”促销海报，整体红色火热风格，金币、红包、火焰元素，右下有二维码和喇叭。\n\n可识别文字（OCR）如下：\n- 爆款课程真底价 · 价保全年\n- 双十一火爆开启\n- 求职 | 晋升 | 论文 | 项目落地一站式速通\n- 抢占优惠&详情咨询，请咨询你的专属助教\n\n需要我：\n- 导出为可复制文案/翻译成英文\n- 改写更高转化的标题与CTA\n- 给出版式与配色优化建议，或生成多版本文案\n- 适配不同平台尺寸（如朋友圈、横版、电商主图）吗？\n\n如果你说下目标人群和投放渠道，我可以给出更精准的文案与设计建议。'}]}]

In [67]:
total_tokens = await token_counter.count(truncated_msgs)

In [68]:
total_tokens

205

&emsp;&emsp;此外，AgentScope还支持自定义消息裁剪器、支持持久化记忆存储以及与Mem0、ReMe等项目进行协同长期记忆管理等功能。

### <center>5. AgentScope接入工具流程

- 查看AgentScope内置工具

In [69]:
import agentscope.tool

In [70]:
print("内置工具函数：")
for _ in agentscope.tool.__all__:
    if _ not in ["Toolkit", "ToolResponse"]:
        print(_)

内置工具函数：
execute_python_code
execute_shell_command
view_text_file
write_text_file
insert_text_file
dashscope_text_to_image
dashscope_text_to_audio
dashscope_image_to_text
openai_text_to_image
openai_text_to_audio
openai_edit_image
openai_create_image_variation
openai_image_to_text
openai_audio_to_text


- 接入自定义查询天气工具

> 这里需要先登录openweather官网获取API-KEY：https://home.openweathermap.org/ ，然后将其写入.env中的`OPENWEATHER_API_KEY`变量中。

In [71]:
import os
from dotenv import load_dotenv 
load_dotenv(override=True)

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [72]:
import requests,json

In [73]:
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Step 2.设置查询参数
    params = {
        "q": loc,               
        "appid": os.getenv("OPENWEATHER_API_KEY"),    # 输入API key
        "units": "metric",            # 使用摄氏度而不是华氏度
        "lang":"zh_cn"                # 输出语言为简体中文
    }

    # Step 3.发送GET请求
    response = requests.get(url, params=params)
    
    # Step 4.解析响应
    data = response.json()
    return json.dumps(data)

In [74]:
get_weather("Beijing")

'{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "\\u9634\\uff0c\\u591a\\u4e91", "icon": "04n"}], "base": "stations", "main": {"temp": 8.94, "feels_like": 8.25, "temp_min": 8.94, "temp_max": 8.94, "pressure": 1020, "humidity": 34, "sea_level": 1020, "grnd_level": 1015}, "visibility": 10000, "wind": {"speed": 1.68, "deg": 212, "gust": 2.12}, "clouds": {"all": 100}, "dt": 1762772876, "sys": {"type": 1, "id": 9609, "country": "CN", "sunrise": 1762728798, "sunset": 1762765404}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}'

紧接着将其封装为AgentScope能够识别的外部函数：

In [75]:
from agentscope.tool import ToolResponse
from agentscope.message import TextBlock
import requests, os, json

def get_weather(loc: str) -> ToolResponse:
    """查询即时天气函数。

    Args:
        loc (str):
            查询天气的城市名称（例如 'Beijing'、'New York'）。
    """
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": loc,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn",
    }

    response = requests.get(url, params=params)
    data = response.json()

    return ToolResponse(
        content=[TextBlock(type="text", text=json.dumps(data, ensure_ascii=False))]
    )

> 注意这是一种Docstring（Documentation String）的函数编写方法，是 Python 官方定义的“文档注释标准”，用于描述函数、类、模块、方法的用途、参数、返回值、异常等信息。

AgentScope 会自动从 docstring 中解析 JSON Schema，无需手动编写。但是需要手动进行工具注册：

In [84]:
from agentscope.tool import Toolkit

In [86]:
# toolkit.clear()

In [87]:
toolkit = Toolkit()
toolkit.register_tool_function(get_weather)

In [88]:
import json
print(json.dumps(toolkit.get_json_schemas(), indent=2, ensure_ascii=False))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "parameters": {
        "properties": {
          "loc": {
            "description": "查询天气的城市名称（例如 'Beijing'、'New York'）。",
            "type": "string"
          }
        },
        "required": [
          "loc"
        ],
        "type": "object"
      },
      "description": "查询即时天气函数。"
    }
  }
]


In [89]:
weather_agent = ReActAgent(
    name="WeatherAgent",
    sys_prompt="你是一个名助人为乐的天气查询助手",
    model=model,
    formatter=OpenAIChatFormatter(),
    memory=InMemoryMemory(),
    toolkit=toolkit
)

In [90]:
msg = Msg(
    name="user1",
    content="请问北京今天天气如何？",
    role="user",
)

In [91]:
res1 = await weather_agent(msg)

WeatherAgent: {
    "type": "tool_use",
    "id": "call_1dfYJrLviijGxZAIzXdlcTw0",
    "name": "get_weather",
    "input": {
        "loc": "Beijing"
    }
}
system: {
    "type": "tool_result",
    "id": "call_1dfYJrLviijGxZAIzXdlcTw0",
    "name": "get_weather",
    "output": [
        {
            "type": "text",
            "text": "{\"coord\": {\"lon\": 116.3972, \"lat\": 39.9075}, \"weather\": [{\"id\": 804, \"main\": \"Clouds\", \"description\": \"阴，多云\", \"icon\": \"04n\"}], \"base\": \"stations\", \"main\": {\"temp\": 8.94, \"feels_like\": 8.25, \"temp_min\": 8.94, \"temp_max\": 8.94, \"pressure\": 1020, \"humidity\": 34, \"sea_level\": 1020, \"grnd_level\": 1015}, \"visibility\": 10000, \"wind\": {\"speed\": 1.68, \"deg\": 212, \"gust\": 2.12}, \"clouds\": {\"all\": 100}, \"dt\": 1762772876, \"sys\": {\"type\": 1, \"id\": 9609, \"country\": \"CN\", \"sunrise\": 1762728798, \"sunset\": 1762765404}, \"timezone\": 28800, \"id\": 1816670, \"name\": \"Beijing\", \"cod\": 200}"
  

In [92]:
messages = await weather_agent.memory.get_memory()

In [93]:
messages

[Msg(id='hHdAarsjccg7bW2SejNpe8', name='user1', content='请问北京今天天气如何？', role='user', metadata=None, timestamp='2025-11-10 19:19:41.414', invocation_id='None'),
 Msg(id='Ctjurc8n9FPBfP6Kjde4rr', name='WeatherAgent', content=[{'type': 'tool_use', 'id': 'call_1dfYJrLviijGxZAIzXdlcTw0', 'name': 'get_weather', 'input': {'loc': 'Beijing'}}], role='assistant', metadata=None, timestamp='2025-11-10 19:19:58.033', invocation_id='None'),
 Msg(id='HxKXXQodWLm4G3YzpcG82Z', name='system', content=[{'type': 'tool_result', 'id': 'call_1dfYJrLviijGxZAIzXdlcTw0', 'name': 'get_weather', 'output': [{'type': 'text', 'text': '{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "阴，多云", "icon": "04n"}], "base": "stations", "main": {"temp": 8.94, "feels_like": 8.25, "temp_min": 8.94, "temp_max": 8.94, "pressure": 1020, "humidity": 34, "sea_level": 1020, "grnd_level": 1015}, "visibility": 10000, "wind": {"speed": 1.68, "deg": 212, "gust": 2.12}, "clouds": {"all":

&emsp;&emsp;能够看出这就是一个典型的Function calling流程。一次完整的`Function calling`执行流程如下：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250318202029130.png" alt="image-20250318202029130" style="zoom:35%;" />

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/202412191720637.png" alt="202412191720637" style="zoom:50%;" />

同时也满足一个React Agent的基本运行流程：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028154837987.png" alt="image-20251028154837987" style="zoom:50%;" />

更多大模型底层调用工具原理、流程、Fucntion calling原理介绍，对更深度内容感兴趣的同学欢迎报名[《2025大模型Agent智能体开发实战》(秋季班)](https://ix9mq.xetslk.com/s/MENek)付费课程进行学习。

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/47b02c2f08aab954e731331395195315.png" alt="b3a518f1a9821408a79363cf694f5172" style="zoom: 15%;" />

而对于React Agent来说，其工具调用的核心逻辑也是React循环工具调用，即可以在简短的推理步骤和有针对性的工具调用之间交替，并将得到的观察结果反馈到后续决策中，直到他们能够给出最终答案。并且具备如下特性：
- 按顺序调用多个工具（由单个提示触发）
- 适当时并行调用工具
- 根据先前结果进行动态工具选择
- 工具重试逻辑和错误处理
- 跨工具调用的状态持久性

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110192334067.png" alt="image-20251110192334067" style="zoom:50%;" />

In [94]:
msg = Msg(
    name="user1",
    content="好的，北京有点冷，那南昌和杭州哪里更暖和一些呢？",
    role="user",
)

In [95]:
await weather_agent(msg)

WeatherAgent: {
    "type": "tool_use",
    "id": "call_2TGceHaRnktfi4uVFE6r6Mur",
    "name": "get_weather",
    "input": {
        "loc": "Nanchang"
    }
}
WeatherAgent: {
    "type": "tool_use",
    "id": "call_RabmI1utkIMHKVZFCfUDkij4",
    "name": "get_weather",
    "input": {
        "loc": "Hangzhou"
    }
}
system: {
    "type": "tool_result",
    "id": "call_2TGceHaRnktfi4uVFE6r6Mur",
    "name": "get_weather",
    "output": [
        {
            "type": "text",
            "text": "{\"coord\": {\"lon\": 115.8833, \"lat\": 28.6833}, \"weather\": [{\"id\": 804, \"main\": \"Clouds\", \"description\": \"阴，多云\", \"icon\": \"04n\"}], \"base\": \"stations\", \"main\": {\"temp\": 17.52, \"feels_like\": 17.26, \"temp_min\": 17.52, \"temp_max\": 17.52, \"pressure\": 1017, \"humidity\": 74, \"sea_level\": 1017, \"grnd_level\": 1007}, \"visibility\": 10000, \"wind\": {\"speed\": 4.18, \"deg\": 353, \"gust\": 8.4}, \"clouds\": {\"all\": 100}, \"dt\": 1762773897, \"sys\": {\"country\": 

Msg(id='VcZXhYYG9qSZj5eKhPMggg', name='WeatherAgent', content='目前南昌更暖和一些：\n- 南昌：约 17.5°C（体感 17.3°C），湿度 74%，偏北风 4.2 m/s\n- 杭州：约 17.0°C（体感 16.4°C），湿度 65%，东风 3.5 m/s\n\n两地温差不大，但南昌略暖，体感也稍占优。相比北京的约 9°C，南昌和杭州都暖和不少。建议穿薄外套/长袖即可。\n\n需要我对比两地接下来几小时或明天的气温与降水吗？', role='assistant', metadata=None, timestamp='2025-11-10 19:25:03.657', invocation_id='None')

In [96]:
await weather_agent.memory.get_memory()

[Msg(id='hHdAarsjccg7bW2SejNpe8', name='user1', content='请问北京今天天气如何？', role='user', metadata=None, timestamp='2025-11-10 19:19:41.414', invocation_id='None'),
 Msg(id='Ctjurc8n9FPBfP6Kjde4rr', name='WeatherAgent', content=[{'type': 'tool_use', 'id': 'call_1dfYJrLviijGxZAIzXdlcTw0', 'name': 'get_weather', 'input': {'loc': 'Beijing'}}], role='assistant', metadata=None, timestamp='2025-11-10 19:19:58.033', invocation_id='None'),
 Msg(id='HxKXXQodWLm4G3YzpcG82Z', name='system', content=[{'type': 'tool_result', 'id': 'call_1dfYJrLviijGxZAIzXdlcTw0', 'name': 'get_weather', 'output': [{'type': 'text', 'text': '{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 804, "main": "Clouds", "description": "阴，多云", "icon": "04n"}], "base": "stations", "main": {"temp": 8.94, "feels_like": 8.25, "temp_min": 8.94, "temp_max": 8.94, "pressure": 1020, "humidity": 34, "sea_level": 1020, "grnd_level": 1015}, "visibility": 10000, "wind": {"speed": 1.68, "deg": 212, "gust": 2.12}, "clouds": {"all":

In [97]:
from datetime import datetime
import os

In [98]:
def write_file(content: str) -> ToolResponse:
    """将指定内容写入本地文件。

    Args:
        content (str): 需要写入的文档内容。

    Returns:
        ToolResponse: 写入结果的反馈消息。
    """
    try:
        # Step 1️⃣ 生成唯一文件名
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"output_{timestamp}.txt"

        # Step 2️⃣ 写入文件
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)

        abs_path = os.path.abspath(filename)
        result_msg = f"✅ 已成功写入本地文件：{abs_path}"

    except Exception as e:
        result_msg = f"❌ 文件写入失败：{str(e)}"

    # Step 3️⃣ 返回标准 ToolResponse
    return ToolResponse(
        content=[
            TextBlock(type="text", text=result_msg)
        ]
    )

In [103]:
toolkit.clear()

In [104]:
toolkit = Toolkit()
toolkit.register_tool_function(get_weather)
toolkit.register_tool_function(write_file)

In [105]:
import json
print(json.dumps(toolkit.get_json_schemas(), indent=2, ensure_ascii=False))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "parameters": {
        "properties": {
          "loc": {
            "description": "查询天气的城市名称（例如 'Beijing'、'New York'）。",
            "type": "string"
          }
        },
        "required": [
          "loc"
        ],
        "type": "object"
      },
      "description": "查询即时天气函数。"
    }
  },
  {
    "type": "function",
    "function": {
      "name": "write_file",
      "parameters": {
        "properties": {
          "content": {
            "description": "需要写入的文档内容。",
            "type": "string"
          }
        },
        "required": [
          "content"
        ],
        "type": "object"
      },
      "description": "将指定内容写入本地文件。"
    }
  }
]


In [106]:
multi_tool_agent = ReActAgent(
    name="MultiToolAgent",
    sys_prompt="你是一个名助人为乐的助手，能够查询天气和进行文本的本地文件写入",
    model=model,
    formatter=OpenAIChatFormatter(),
    memory=InMemoryMemory(),
    toolkit=toolkit
)

In [107]:
msg = Msg(
    name="user1",
    content="请帮我查询天津、石家庄、上海等地天气，并写入本地文件。",
    role="user",
)

In [108]:
await multi_tool_agent(msg)

MultiToolAgent: {
    "type": "tool_use",
    "id": "call_GY2lanRnMSmFsUkY2xAFrumW",
    "name": "get_weather",
    "input": {
        "loc": "天津"
    }
}
MultiToolAgent: {
    "type": "tool_use",
    "id": "call_R2iu74GrMS5M7aCnEiyqmry7",
    "name": "get_weather",
    "input": {
        "loc": "石家庄"
    }
}
MultiToolAgent: {
    "type": "tool_use",
    "id": "call_sB3F6hRi7JCdD5YAZRPzbndG",
    "name": "get_weather",
    "input": {
        "loc": "上海"
    }
}
system: {
    "type": "tool_result",
    "id": "call_GY2lanRnMSmFsUkY2xAFrumW",
    "name": "get_weather",
    "output": [
        {
            "type": "text",
            "text": "{\"cod\": \"404\", \"message\": \"city not found\"}"
        }
    ]
}
system: {
    "type": "tool_result",
    "id": "call_R2iu74GrMS5M7aCnEiyqmry7",
    "name": "get_weather",
    "output": [
        {
            "type": "text",
            "text": "{\"cod\": \"404\", \"message\": \"city not found\"}"
        }
    ]
}
system: {
    "type": "tool_

Msg(id='5uCCCr7t6B3StJmz4V5wzQ', name='MultiToolAgent', content='我已为您查询以下城市的当前天气，并写入到本地文件：\n\n- 天津：阴，多云；9.97°C；湿度 71%；气压 1020 hPa；风 1.00 m/s（160°）；能见度 8000 m；云量 93%\n- 石家庄：阴，多云；13.72°C；湿度 43%；气压 1020 hPa；风 1.04 m/s（124°，阵风 1.54 m/s）；能见度 10000 m；云量 100%\n- 上海：晴；15.92°C；湿度 59%；气压 1021 hPa；风 4.00 m/s（50°）；能见度 10000 m；云量 0%\n\n文件已保存到：E:\\work\\大模型课程课件\\公开课直播项目\\251110AgentScope\\output_20251110_193133.txt\n如果您需要追加更多城市或改为特定格式（如 CSV/Markdown），我可以继续处理。', role='assistant', metadata=None, timestamp='2025-11-10 19:31:42.037', invocation_id='None')

In [109]:
await multi_tool_agent.memory.get_memory()

[Msg(id='Bg6Hebqus5UKrMRW94T6F8', name='user1', content='请帮我查询天津、石家庄、上海等地天气，并写入本地文件。', role='user', metadata=None, timestamp='2025-11-10 19:30:11.598', invocation_id='None'),
 Msg(id='RN4Jhr9QkfDQ8hH6gM7GvG', name='MultiToolAgent', content=[{'type': 'tool_use', 'id': 'call_GY2lanRnMSmFsUkY2xAFrumW', 'name': 'get_weather', 'input': {'loc': '天津'}}, {'type': 'tool_use', 'id': 'call_R2iu74GrMS5M7aCnEiyqmry7', 'name': 'get_weather', 'input': {'loc': '石家庄'}}, {'type': 'tool_use', 'id': 'call_sB3F6hRi7JCdD5YAZRPzbndG', 'name': 'get_weather', 'input': {'loc': '上海'}}], role='assistant', metadata=None, timestamp='2025-11-10 19:30:29.239', invocation_id='None'),
 Msg(id='KdDqHREmXbHoDPypsq8tNJ', name='system', content=[{'type': 'tool_result', 'id': 'call_GY2lanRnMSmFsUkY2xAFrumW', 'name': 'get_weather', 'output': [{'type': 'text', 'text': '{"cod": "404", "message": "city not found"}'}]}], role='system', metadata=None, timestamp='2025-11-10 19:30:29.255', invocation_id='None'),
 Msg(id='nEXkHHuZ4W

查询记录结果如下：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251110193356185.png" alt="image-20251110193356185" style="zoom:33%;" />

In [ ]:
xiaozhi = ReActAgent(
    name="小智",
    sys_prompt="你是一个名为小智的智能助手",
    model=model,
    formatter=OpenAIChatFormatter(),
    memory=InMemoryMemory(),
)

In [45]:
serialized_msg = msg.to_dict()

In [46]:
serialized_msg

{'id': '97xqKvRsFBTh2r2zGnh8tq',
 'name': 'User',
 'role': 'user',
 'content': [{'type': 'text', 'text': '请帮我详细描述图片上的内容'},
  {'type': 'image',
   'source': {'type': 'url',
    'url': 'https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/ffc998ffffe1af118077b4c51dc75148.png'}}],
 'metadata': None,
 'timestamp': '2025-11-10 18:06:39.444'}

In [47]:
model = DashScopeChatModel(
    model_name="qwen-max",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    stream=False,
)

In [48]:
res = await model(messages = [serialized_msg])

In [49]:
res

ChatResponse(content=[{'type': 'text', 'text': '当然可以帮助您，但是我需要您先上传或描述一下图片的内容。由于当前我无法直接查看图片，您可以告诉我图片中有什么，或者尝试通过文字描述您希望了解的图片内容，我会尽我所能提供帮助。如果您能分享图片的具体信息，比如其中的人物、场景、颜色等细节，那就更好了。'}], id='2025-11-10 18:06:58.218_f1ec75', created_at='2025-11-10 18:06:58.218', type='chat', usage=ChatUsage(input_tokens=15, output_tokens=69, time=3.724085, type='chat'), metadata=None)